In [18]:

import requests
from bs4 import BeautifulSoup
import re


urls = [
    "https://en.wikipedia.org/wiki/Retrieval-augmented_generation",
    "https://en.wikipedia.org/wiki/OpenAI",
    "https://en.wikipedia.org/wiki/Generative_pre-trained_transformer",
    "https://en.wikipedia.org/wiki/Natural_language_processing",
    "https://en.wikipedia.org/wiki/Vector_database",
    "https://en.wikipedia.org/wiki/Information_retrieval",
    "https://en.wikipedia.org/wiki/Transformer_(machine_learning_model)",
    "https://en.wikipedia.org/wiki/Neural_network",
    "https://en.wikipedia.org/wiki/Deep_learning",
    "https://en.wikipedia.org/wiki/Machine_learning",
    "https://en.wikipedia.org/wiki/Knowledge_graph",
    "https://en.wikipedia.org/wiki/Information_extraction",
    "https://en.wikipedia.org/wiki/FAISS",
    "https://en.wikipedia.org/wiki/AI_alignment"
]
     

In [19]:
import time
import random

def clean_text(content):
    content = re.sub(r'\[\d+\]', '', content) 
    content = re.sub(r'\s+', ' ', content)    
    return content.strip()

def fetch_and_clean(url, session, max_retries=3):
    """Fetch and clean Wikipedia article with retry logic"""
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    for attempt in range(max_retries):
        try:
            response = session.get(url, headers=headers, timeout=15)
            response.raise_for_status()
            
            soup = BeautifulSoup(response.content, 'html.parser')
            content = soup.find('div', {'class': 'mw-parser-output'})
            
            if not content:
                content = soup.find('div', {'id': 'mw-content-text'})
            
            if not content:
                content = soup.find('main', {'role': 'main'})
            
            if not content:
                print(f"    DEBUG: No content div found. Response length: {len(response.content)}")
                if attempt < max_retries - 1:
                    time.sleep(2)
                    continue
                return ""
            
            for section_title in ['References', 'Bibliography', 'External links', 'See also', 'Notes']:
                section = content.find(id=section_title.replace(" ", "_"))
                if section:
                    parent = section.find_parent(['h2', 'h3', 'h4'])
                    if parent:
                        for sibling in parent.find_next_siblings():
                            if sibling.name in ['h2', 'h3', 'h4']:
                                break
                            sibling.decompose()
                        parent.decompose()
            
            for tag in content(['script', 'style']):
                tag.decompose()
            
            text = content.get_text(separator='\n', strip=True)
            text = clean_text(text)
            
            return text
            
        except requests.exceptions.HTTPError as e:
            if e.response.status_code in [429, 403]:
                wait_time = (2 ** attempt) + random.uniform(0, 1)
                print(f"    Rate limited. Waiting {wait_time:.1f}s before retry...")
                time.sleep(wait_time)
            else:
                raise
        except Exception as e:
            print(f"    Exception (attempt {attempt+1}/{max_retries}): {str(e)[:60]}")
            if attempt == max_retries - 1:
                raise
            time.sleep(2)
    
    return ""

session = requests.Session()

with open('Data_from_wiki.txt', 'w', encoding='utf-8') as file:
    for idx, url in enumerate(urls, 1):
        try:
            print(f"[{idx}/{len(urls)}] Fetching: {url}")
            clean_article_text = fetch_and_clean(url, session)
            
            if clean_article_text and len(clean_article_text) > 100:
                file.write(clean_article_text)
                file.write("\n\n")
                success_count += 1
                print(f"  OK ({len(clean_article_text)} chars)")
            else:
                print(f"  EMPTY ({len(clean_article_text) if clean_article_text else 0} chars)")
                empty_count += 1
                
        except Exception as e:
            print(f"  FAILED: {str(e)[:80]}")
            failed_count += 1
        
        # Longer delay
        if idx < len(urls):
            time.sleep(random.uniform(4, 6))

session.close()


[1/14] Fetching: https://en.wikipedia.org/wiki/Retrieval-augmented_generation
  OK (20278 chars)
[2/14] Fetching: https://en.wikipedia.org/wiki/OpenAI
  OK (111058 chars)
[3/14] Fetching: https://en.wikipedia.org/wiki/Generative_pre-trained_transformer
  OK (39545 chars)
[4/14] Fetching: https://en.wikipedia.org/wiki/Natural_language_processing
  OK (51173 chars)
[5/14] Fetching: https://en.wikipedia.org/wiki/Vector_database
  OK (16760 chars)
[6/14] Fetching: https://en.wikipedia.org/wiki/Information_retrieval
  OK (36971 chars)
[7/14] Fetching: https://en.wikipedia.org/wiki/Transformer_(machine_learning_model)
  OK (97499 chars)
[8/14] Fetching: https://en.wikipedia.org/wiki/Neural_network
  OK (6239 chars)
[9/14] Fetching: https://en.wikipedia.org/wiki/Deep_learning
  OK (135666 chars)
[10/14] Fetching: https://en.wikipedia.org/wiki/Machine_learning
  OK (121450 chars)
[11/14] Fetching: https://en.wikipedia.org/wiki/Knowledge_graph
  OK (20277 chars)
[12/14] Fetching: https://en.wik